# Tool calling
Models 会请求能够完成任务的 tools，比如从一个数据库中拉取数据，搜索网页，运行代码。
一个工具（Tool/Skill）必须包含两部分：
1. 描述信息（Schema）工具叫什么名字、干什么的、需要什么参数（JSON Schema）→ 给模型看的
2. 真正执行的函数：模型调用后，真正跑代码的地方 → 给程序执行用的

在大模型行业里：
OpenAI / LangChain 叫：Tool（工具）
企业里 / 业务口语 叫：Skill（技能）
技术底层 叫：Function Call（函数调用）

## 单个工具调用
为了让 model 可以利用到定义好的 tools，必须使用 bind_tools 绑定，在随后的调用中，可以按需调用。

注意：
当你给模型 model 绑定工具后，模型 model 返回的不是答案，而是「我要调用某个工具」的请求。
    模型绑定工具后 → 模型不直接回答你！
    模型只会说：
    我需要调用工具 get_weather，参数是深圳

In [ ]:
import os

from langchain.tools import tool
from langchain.chat_models import init_chat_model


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    # 用 @tool 装饰器定义的 get_weather，已经被 LangChain 封装成了 StructuredTool 对象
    # 不能直接用 get_weather() 这种函数调用方式。
    # 手动执行工具时，必须通过工具对象的 .invoke() 方法来调用，而不是直接把它当函数执行。
    return f"It's sunny in {location}."


model = init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

model_with_tools = model.bind_tools([get_weather])
func_calls = model_with_tools.invoke("What is the weather in Shenzhen?")
print(func_calls)

for tool_call in func_calls.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 275, 'total_tokens': 349, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 43, 'rejected_prediction_tokens': None, 'text_tokens': 74}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 275}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-957e4d5a-9733-9d2d-8a76-1df3e450f48c', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d56fb-b352-7a71-8a79-65b4b5398b6d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Shenzhen'}, 'id': 'call_a704ba67745e4910af15567e', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 275, 'output_tokens': 74, 'total_tokens': 349, 'input_token_details': {}, 'output_token_details': {'reasoning': 43}}
Tool: get_weather
Args: {'location': 'Shenzhe

如果你只是单独用模型 model ，没有用 Agent，那必须你自己手动执行工具，再把结果传回模型继续推理。
    如果你不用 Agent（只用 model + tool）
    你必须自己写代码：
        看模型要不要调用工具
        如果要，手动执行工具
        把工具结果再发给模型
        让模型最后生成答案
    👉 等于你自己当代理人，手动跑腿。

    当一个模型model 返回方法调用了，你需要将这个执行这个 tools,并且将结果返回给到model.才能创建对话循环，model 才能用这个 tool 生成最终结果，或者使用 LangChain 中封装好的 Agent


In [ ]:
# 你必须手动执行
tool_name = func_calls.tool_calls[0]["name"]
args = func_calls.tool_calls[0]["args"]

result = get_weather.invoke(args)

# 再手动把结果发回模型（注意：tool消息必须是ToolMessage格式）
from langchain_core.messages import ToolMessage

final = model.invoke([
    ("user", "深圳天气？"),
    func_calls,  # 直接传入模型返回的AIMessage，比手动写更规范
    ToolMessage(content=result, tool_call_id=func_calls.tool_calls[0]["id"])
])

print(final.content)

深圳目前天气晴朗。


In [ ]:
messages = [{"role": "user", "content": "深圳的天气怎么样？"}]

# Model generates tool calls
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

深圳现在是晴天。


但如果你用了 LangChain 封装好的 Agent，那「自动调用工具 → 拿结果 → 继续思考」这个循环，Agent 会自动帮你做完！
    如果你用 Agent
    你什么都不用管！
    Agent 自动干：
    看模型要不要调用工具
    自动执行工具
    自动把结果传回模型
    自动继续思考
    直到最后给出答案
    👉 Agent = 全自动代理人，不用你管流程！

In [17]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

# 绑定工具
tools = [get_weather]
model_with_tools = model.bind_tools(tools)

agent = create_agent(model_with_tools, tools)

# 一句话 → 自动调用工具 → 自动返回答案
result = agent.invoke({"messages": [("user", "深圳天气怎么样？")]},)
for msg in result["messages"]:
    if msg.content:
        print(msg.content)

深圳天气怎么样？
It's sunny in 深圳.
深圳今天天气晴朗，阳光明媚。


## 并行工具调用
在合适的时候，很多模型支持并行调用多个工具，使得 model 可以同时汇集从不同源来的信息。

In [20]:
response = model_with_tools.invoke("深圳和北京的天气分别怎么样？")
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_c429483b19824311b3336ff9', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_74a43bcd2e19409f91a06a85', 'type': 'tool_call'}]


In [22]:
results = []
for tool_call in response.tool_calls:
    if tool_call['name'] == "get_weather":
        result = get_weather.invoke(tool_call)

    results.append(result)

print(results)

[ToolMessage(content="It's sunny in 深圳.", name='get_weather', tool_call_id='call_c429483b19824311b3336ff9'), ToolMessage(content="It's sunny in 北京.", name='get_weather', tool_call_id='call_74a43bcd2e19409f91a06a85')]


## 流媒体工具调用
是流式响应时，工具调用会逐渐的构建通过 ToolCallChunk。
你会看到工具调用的过程当在生成的时候，而不是等待完成回答完成时。

In [23]:
for chunk in model_with_tools.stream(
    "深圳和北京的天气分别怎么样？"
):
    for tool_chunk in chunk.tool_call_chunks:
        if name := tool_chunk.get("name"):
            print(f"Tool:{name}")
        if id_ := tool_chunk.get("id"):
            print(f"ID: {id_}")
        if args := tool_chunk.get("args"):
            print(f"Args: {args}")

Tool:get_weather
ID: call_5cb94f278b6b48f9bef3a18d
Args: {"location": 
Args: "深圳
Args: "
Args: }
Tool:get_weather
ID: call_168fcbe755554d489af923f0
Args: {"location": "北京
Args: "
Args: }


可以积累各个块来完整构建工具调用

In [ ]:
gathered = None
for chunk in model_with_tools.stream("深圳和北京的天气分别怎么样？"):
    gathered = chunk if gathered is None else gathered + chunk
    print(gathered.tool_calls)

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[{'name': 'get_weather', 'args': {}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {}, 'id': 'call_988bb8404b91430cb6141bdf', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': {'location': '深圳'}, 'id': 'call_dc321f1a138f4334aae9f0ee', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': '北京'}, 'id': 'call_988bb8404b91430cb6141bdf', 'type':